# Name: Baraka Mandi

## Coding Problem 2: Vision Transformer Patch Embedding

**Objective:** Implement the input embedding component of a Vision Transformer (ViT) in PyTorch.

| Parameter | Value |
|---|---|
| Input image size | 32 × 32 |
| Image channels | 3 (RGB) |
| Patch size | 8 × 8 |
| Embedding dimension | 64 |
| Batch size | 2 |

With these settings each image yields $N = (32/8)^2 = 16$ patches, each flattened patch has $P \times P \times C = 8 \cdot 8 \cdot 3 = 192$ values, and the final sequence length is $N + 1 = 17$.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# Model configuration
IMAGE_SIZE = 32
CHANNELS = 3
PATCH_SIZE = 8
EMBED_DIM = 64
BATCH_SIZE = 2

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.13.0


### (a) Image Patch Extraction

In [4]:
def extract_patches(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    # Split a batch of images into flattened, non-overlapping patches.
    #   images:     tensor of shape (B, C, H, W)
    #   patch_size: side length P of each square patch
    #   returns:    tensor of shape (B, N, P*P*C), N = (H // P) * (W // P)
    if images.dim() != 4:
        raise ValueError(f"Expected a 4D tensor (B, C, H, W), got shape {tuple(images.shape)}")
    B, C, H, W = images.shape
    P = patch_size
    if H % P != 0 or W % P != 0:
        raise ValueError(f"Image size ({H}x{W}) must be divisible by patch size {P}")

    patches = images.unfold(2, P, P).unfold(3, P, P)       # (B, C, H/P, W/P, P, P)
    patches = patches.permute(0, 2, 3, 4, 5, 1)            # (B, H/P, W/P, P, P, C)
    num_patches = (H // P) * (W // P)
    return patches.reshape(B, num_patches, P * P * C)      # (B, N, P*P*C)


# Test with random images of shape (2, 3, 32, 32)
images = torch.randn(BATCH_SIZE, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
patches = extract_patches(images, PATCH_SIZE)

print(f"Input image batch shape: {tuple(images.shape)}")
print(f"Extracted patches shape: {tuple(patches.shape)}")

Input image batch shape: (2, 3, 32, 32)
Extracted patches shape: (2, 16, 192)


**Correctness check.** Rebuild a few patches by slicing the image directly and confirm they match the output of `extract_patches`. Patch index $k$ sits at grid row $k \,//\, 4$ and column $k \bmod 4$.

In [5]:
grid = IMAGE_SIZE // PATCH_SIZE  # 4 patches per row / column

def manual_patch(img, k, P=PATCH_SIZE):
    r, c = divmod(k, grid)
    tile = img[:, r * P:(r + 1) * P, c * P:(c + 1) * P]   # (C, P, P)
    return tile.permute(1, 2, 0).reshape(-1)               # flatten in P x P x C order

all_match = all(
    torch.equal(patches[b, k], manual_patch(images[b], k))
    for b in range(BATCH_SIZE)
    for k in range(grid * grid)
)
print(f"Every patch matches direct slicing of the image: {all_match}")
assert all_match

Every patch matches direct slicing of the image: True


### (b) Patch Embedding and Positional Encoding


In [6]:
class ViTPatchEmbedding(nn.Module):
    # Converts a batch of images into the ViT input sequence:
    # [CLS] + linearly projected patches, plus learnable positional embeddings.

    def __init__(self, image_size=32, patch_size=8, in_channels=3, embed_dim=64):
        super().__init__()
        if image_size % patch_size != 0:
            raise ValueError("image_size must be divisible by patch_size")

        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_dim = patch_size * patch_size * in_channels

        # 1. Learnable linear projection of flattened patches
        self.projection = nn.Linear(self.patch_dim, embed_dim)

        # 2. Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # 4. Learnable positional embeddings for N patches + 1 [CLS] token
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

    def forward(self, images, return_patch_embeddings=False):
        B = images.shape[0]

        patches = extract_patches(images, self.patch_size)        # (B, N, P*P*C)
        patch_embeddings = self.projection(patches)                # (B, N, D)

        # 3. Prepend [CLS] token (one copy per image in the batch)
        cls_tokens = self.cls_token.expand(B, -1, -1)              # (B, 1, D)
        x = torch.cat([cls_tokens, patch_embeddings], dim=1)       # (B, N+1, D)

        # 4. Add positional embeddings to the entire sequence
        x = x + self.pos_embedding                                 # (B, N+1, D)

        if return_patch_embeddings:
            return x, patch_embeddings
        return x


vit_embed = ViTPatchEmbedding(IMAGE_SIZE, PATCH_SIZE, CHANNELS, EMBED_DIM)
print(vit_embed)
print()
for name, p in vit_embed.named_parameters():
    print(f"{name:20s} {str(tuple(p.shape)):12s} learnable={p.requires_grad}")
print(f"\nTotal learnable parameters: {sum(p.numel() for p in vit_embed.parameters()):,}")

ViTPatchEmbedding(
  (projection): Linear(in_features=192, out_features=64, bias=True)
)

cls_token            (1, 1, 64)   learnable=True
pos_embedding        (1, 17, 64)  learnable=True
projection.weight    (64, 192)    learnable=True
projection.bias      (64,)        learnable=True

Total learnable parameters: 13,504


### (c) Testing and Verification

Run a batch of two random RGB images through the module and print the required dimensions.

In [7]:
test_images = torch.randn(BATCH_SIZE, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

final_embeddings, patch_embeddings = vit_embed(test_images, return_patch_embeddings=True)

num_patches = vit_embed.num_patches
patch_dim = vit_embed.patch_dim

print(f"1. Number of patches per image:              {num_patches}")
print(f"2. Flattened patch dimension (P x P x C):    {patch_dim}")
print(f"3. Patch embedding shape (before [CLS]):     {tuple(patch_embeddings.shape)}")
print(f"4. Final embedding shape ([CLS] + pos emb):  {tuple(final_embeddings.shape)}")

1. Number of patches per image:              16
2. Flattened patch dimension (P x P x C):    192
3. Patch embedding shape (before [CLS]):     (2, 16, 64)
4. Final embedding shape ([CLS] + pos emb):  (2, 17, 64)


In [8]:
# Verify the shapes against the expected values
expected_N = (IMAGE_SIZE // PATCH_SIZE) ** 2
expected_patch_dim = PATCH_SIZE * PATCH_SIZE * CHANNELS
expected_final = (BATCH_SIZE, expected_N + 1, EMBED_DIM)

checks = {
    "Number of patches == 16": num_patches == expected_N == 16,
    "Flattened patch dim == 192": patch_dim == expected_patch_dim == 192,
    "Patch embeddings == (2, 16, 64)": tuple(patch_embeddings.shape) == (BATCH_SIZE, expected_N, EMBED_DIM),
    "Final embeddings == (2, 17, 64)": tuple(final_embeddings.shape) == expected_final,
}

# Position 0 of the output must be the [CLS] token plus positional embedding 0
cls_out = final_embeddings[:, 0, :]
cls_expected = (vit_embed.cls_token + vit_embed.pos_embedding[:, 0:1, :]).squeeze(1).expand(BATCH_SIZE, -1)
checks["Position 0 holds [CLS] + pos_emb[0]"] = torch.allclose(cls_out, cls_expected)

# Positions 1..N must be the patch embeddings plus positional embeddings 1..N
checks["Positions 1..N hold patches + pos_emb[1:]"] = torch.allclose(
    final_embeddings[:, 1:, :], patch_embeddings + vit_embed.pos_embedding[:, 1:, :]
)

# Gradients must flow into every learnable component
final_embeddings.sum().backward()
checks["Gradients reach projection, [CLS], pos emb"] = all(
    p.grad is not None and p.grad.abs().sum() > 0 for p in vit_embed.parameters()
)

for desc, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {desc}")

assert all(checks.values()), "One or more checks failed"
print(f"\nVerified: final embedding shape {tuple(final_embeddings.shape)} == expected (B, N + 1, D) = {expected_final}")

[PASS] Number of patches == 16
[PASS] Flattened patch dim == 192
[PASS] Patch embeddings == (2, 16, 64)
[PASS] Final embeddings == (2, 17, 64)
[PASS] Position 0 holds [CLS] + pos_emb[0]
[PASS] Positions 1..N hold patches + pos_emb[1:]
[PASS] Gradients reach projection, [CLS], pos emb

Verified: final embedding shape (2, 17, 64) == expected (B, N + 1, D) = (2, 17, 64)


## Why ViT needs positional embeddings and a [CLS] token

**Positional embeddings.** Self-attention is permutation-invariant: it treats its input as an unordered set, so shuffling the patches would produce the same set of outputs, just shuffled. Flattening the image into a sequence of patches throws away the information about where each patch came from, yet spatial layout is essential for vision (a sky patch above a grass patch means something different from the reverse). Adding a learnable positional embedding to each token gives every position a unique signature, letting the model learn spatial relationships such as which patches are neighbours, which share a row or column, and where the image centre lies. Because they are learned rather than fixed, ViT discovers a 2D-aware structure on its own; the original paper found that learned 1D positional embeddings end up encoding row and column similarity.

**[CLS] token.** The Transformer encoder outputs one vector per input token, but classification needs a single vector for the whole image. The [CLS] token is a learnable placeholder that holds no image content of its own. Through the encoder layers it attends to every patch and gathers information from all of them, so its final hidden state works as a global image representation that is passed to the classification head. Using a dedicated token (borrowed from BERT) avoids favouring any particular patch and lets the model learn how to pool information, rather than relying on a fixed rule like averaging. It also receives its own positional embedding (position 0), which is why the positional table has $N + 1$ entries.